## Mental Health Platform - Analysis

Looker Studio: https://datastudio.google.com/reporting/b81a6d91-fa00-4427-9050-766fe7d2644f

You can view the above link for the dashboard (It consists of 3 pages)

In [ ]:
#First, I will go with importing the downloaded file and basic libraries.
import pandas as pd
import numpy as np



from google.colab import files
uploaded = files.upload()

Saving SAMPLE_DATA.xlsx to SAMPLE_DATA (1).xlsx


In [ ]:
import pandas as pd
import numpy as np

# I loaded each sheet separately to keep things clear
users_df = pd.read_excel("SAMPLE_DATA.xlsx", sheet_name="Users")
sessions_df = pd.read_excel("SAMPLE_DATA.xlsx", sheet_name="Sessions")
feedback_df = pd.read_excel("SAMPLE_DATA.xlsx", sheet_name="Feedback")

# I wanted to first see where missing values exist
sessions_df.isna().sum()

# fee is numeric, so I decided to fill with median to avoid outliers
sessions_df['fee'] = sessions_df['fee'].fillna(sessions_df['fee'].median())

# therapist_id looked categorical, so I used the most frequent value
common_therapist = sessions_df['therapist_id'].mode()[0]
sessions_df['therapist_id'] = sessions_df['therapist_id'].fillna(common_therapist)

# ratings missing in feedback — filling with median as well
feedback_df['rating'] = feedback_df['rating'].fillna(feedback_df['rating'].median())

Sessions missing values:
session_id        0
user_id           0
session_date      0
session_number    0
therapist_id      0
fee               0
dtype: int64

Feedback missing values:
session_id     0
rating         0
review_text    0
dtype: int64

Users missing values:
user_id        0
signup_date    0
source         0
dtype: int64

After handling — missing values remaining:
session_id        0
user_id           0
session_date      0
session_number    0
therapist_id      0
fee               0
dtype: int64


In [ ]:
# I wanted to quickly check missing values across all sheets
print("Sessions missing values:")
print(sessions_df.isna().sum())

print("\nFeedback missing values:")
print(feedback_df.isna().sum())

print("\nUsers missing values:")
print(users_df.isna().sum())

# there are no major missing values,
# but I still kept basic handling in case some appear later

sessions_df['fee'] = sessions_df['fee'].fillna(sessions_df['fee'].median())

common_therapist = sessions_df['therapist_id'].mode()[0]
sessions_df['therapist_id'] = sessions_df['therapist_id'].fillna(common_therapist)

feedback_df['rating'] = feedback_df['rating'].fillna(feedback_df['rating'].median())

# just rechecking after fill
print("\nAfter handling — missing values remaining:")
print(sessions_df.isna().sum())

Sessions missing values:
session_id        0
user_id           0
session_date      0
session_number    0
therapist_id      0
fee               0
dtype: int64

Feedback missing values:
session_id     0
rating         0
review_text    0
dtype: int64

Users missing values:
user_id        0
signup_date    0
source         0
dtype: int64

After handling — missing values remaining:
session_id        0
user_id           0
session_date      0
session_number    0
therapist_id      0
fee               0
dtype: int64


In [ ]:
# removing duplicates just to be safe
sessions_df = sessions_df.drop_duplicates(subset=['session_id'])
feedback_df = feedback_df.drop_duplicates(subset=['session_id'])
users_df = users_df.drop_duplicates(subset=['user_id'])

# I want to check if session numbers follow 1,2,3... for each user
def check_sequence(group):
    actual_sessions = sorted(group['session_number'].tolist())
    expected_sessions = list(range(1, len(actual_sessions) + 1))
    return actual_sessions == expected_sessions

# applying check per user
sequence_check = sessions_df.groupby('user_id').apply(check_sequence)

# users where sequence looks wrong
invalid_users = sequence_check[~sequence_check].index

print("Users with invalid sequences:", len(invalid_users))

Users with invalid sequences: 0


/tmp/ipykernel_562/3007662550.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sequence_check = sessions_df.groupby('user_id').apply(check_sequence)


In [ ]:
# checking how many sessions each user completed
user_sessions = sessions_df.groupby('user_id')['session_number'].max().reset_index()
user_sessions.rename(columns={'session_number': 'user_lifetime_sessions'}, inplace=True)

# merging sessions with feedback to get avg rating per user
merged_df = sessions_df.merge(feedback_df, on='session_id', how='left')

avg_rating = merged_df.groupby('user_id')['rating'].mean().reset_index()
avg_rating.rename(columns={'rating': 'avg_rating'}, inplace=True)

# converting session date to datetime
sessions_df['session_date'] = pd.to_datetime(sessions_df['session_date'])

# sorting to calculate gap between sessions
sessions_sorted = sessions_df.sort_values(['user_id', 'session_date'])

sessions_sorted['days_between_sessions'] = sessions_sorted.groupby('user_id')['session_date'].diff().dt.days

In [ ]:
# average rating for each therapist
therapist_rating = merged_df.groupby('therapist_id')['rating'].mean().reset_index()

# I counted how many sessions each user completed
user_session_counts = sessions_df.groupby('user_id')['session_id'].count().reset_index()
user_session_counts.rename(columns={'session_id': 'total_sessions'}, inplace=True)

# users with more than one session treated as retained
user_session_counts['retained'] = (user_session_counts['total_sessions'] > 1).astype(int)

# merging with average rating
corr_df = avg_rating.merge(
    user_session_counts[['user_id', 'retained']],
    on='user_id'
)

# I quickly checked correlation between rating and retention
corr_df[['avg_rating', 'retained']].corr()

,avg_rating,retained
avg_rating,1.000000,-0.020859
retained,-0.020859,1.000000


In [ ]:
df = pd.read_excel('SAMPLE_DATA.xlsx')


In [ ]:
#I am importing the SQLITE for SQL queries.
import sqlite3
conn = sqlite3.connect(':memory:')

In [ ]:
# pushing dataframes into sqlite so I can run SQL easily
users_df.to_sql("users", conn, index=False, if_exists="replace")
sessions_df.to_sql("sessions", conn, index=False, if_exists="replace")
feedback_df.to_sql("feedback", conn, index=False, if_exists="replace")

# small helper to run SQL queries
def run_query(sql):
    result = pd.read_sql(sql, conn)
    return result

In [ ]:
# creating cohort based on first session month
cohort_sql = """
SELECT
    user_id,
    strftime('%Y-%m', MIN(session_date)) AS cohort_month
FROM sessions
GROUP BY user_id
"""

cohorts_df = run_query(cohort_sql)

# just checking the output
cohorts_df.head()

,user_id,cohort_month
0,1,2025-04
1,2,2025-07
2,3,2025-04
3,4,2025-01
4,5,2025-04


In [ ]:
# I’m grouping users by the month of their first session
cohort_query = """
SELECT
    user_id,
    strftime('%Y-%m', MIN(session_date)) AS cohort_month
FROM sessions
GROUP BY user_id
"""

cohorts_df = run_query(cohort_query)

# quick look
cohorts_df.head()

,user_id,cohort_month
0,1,2025-04
1,2,2025-07
2,3,2025-04
3,4,2025-01
4,5,2025-04


In [ ]:
# calculating retention for session 2, 3 and 4 by cohort
retention_sql = """
WITH cohort AS (
  SELECT user_id,
         strftime('%Y-%m', MIN(session_date)) AS cohort_month
  FROM sessions
  GROUP BY user_id
),
base AS (
  SELECT cohort_month,
         COUNT(DISTINCT user_id) AS total_users
  FROM cohort
  GROUP BY cohort_month
),
ret AS (
  SELECT c.cohort_month,
    COUNT(DISTINCT CASE WHEN s.session_number = 2 THEN s.user_id END) AS s2,
    COUNT(DISTINCT CASE WHEN s.session_number = 3 THEN s.user_id END) AS s3,
    COUNT(DISTINCT CASE WHEN s.session_number = 4 THEN s.user_id END) AS s4
  FROM sessions s
  JOIN cohort c ON s.user_id = c.user_id
  GROUP BY c.cohort_month
)
SELECT b.cohort_month, b.total_users,
  ROUND(100.0*r.s2/b.total_users,1) AS pct_s2,
  ROUND(100.0*r.s3/b.total_users,1) AS pct_s3,
  ROUND(100.0*r.s4/b.total_users,1) AS pct_s4
FROM base b
JOIN ret r ON b.cohort_month = r.cohort_month
"""

retention_df = run_query(retention_sql)

# I checked the retention output to make sure values look reasonable
retention_df.head()

,cohort_month,total_users,pct_s2,pct_s3,pct_s4
0,2025-01,62,59.7,37.1,14.5
1,2025-02,74,60.8,36.5,23.0
2,2025-03,73,47.9,34.2,21.9
3,2025-04,91,57.1,33.0,18.7
4,2025-05,96,61.5,32.3,16.7


In [ ]:
#I will funnel from session 1 → 2 → 3
funnel_sql = """
WITH counts AS (
  SELECT
    COUNT(DISTINCT user_id) AS total_users,
    COUNT(DISTINCT CASE WHEN session_number >= 1 THEN user_id END) AS s1,
    COUNT(DISTINCT CASE WHEN session_number >= 2 THEN user_id END) AS s2,
    COUNT(DISTINCT CASE WHEN session_number >= 3 THEN user_id END) AS s3
  FROM sessions
)
SELECT
  total_users,
  s1, ROUND(100.0*s1/total_users,1) AS pct_s1,
  s2, ROUND(100.0*s2/s1,1) AS pct_s2,
  s3, ROUND(100.0*s3/s2,1) AS pct_s3
FROM counts
"""

funnel_df = run_query(funnel_sql)
funnel_df.head()

,total_users,s1,pct_s1,s2,pct_s2,s3,pct_s3
0,500,500,100.0,281,56.2,169,60.1


In [ ]:
# total revenue which is summing all fees
sql_total_revenue = """SELECT SUM(fee) AS total_revenue FROM sessions"""
total_revenue_df = run_query(sql_total_revenue)
print("Total Revenue:")
print(total_revenue_df)


# revenue per user
sql_revenue_per_user = """
SELECT user_id, SUM(fee) AS revenue_per_user
FROM sessions
GROUP BY user_id
"""
revenue_per_user_df = run_query(sql_revenue_per_user)
print("\nRevenue Per User:")
print(revenue_per_user_df)


# revenue per cohort, like based on first session month
sql_revenue_per_cohort = """
WITH cohort AS (
  SELECT user_id, strftime('%Y-%m', MIN(session_date)) AS cohort_month
  FROM sessions
  GROUP BY user_id
)
SELECT c.cohort_month, SUM(s.fee) AS revenue
FROM sessions s
JOIN cohort c ON s.user_id = c.user_id
GROUP BY c.cohort_month
"""
revenue_per_cohort_df = run_query(sql_revenue_per_cohort)
print("\nRevenue Per Cohort:")
print(revenue_per_cohort_df)


# revenue per therapist
sql_revenue_per_therapist = """
SELECT therapist_id, SUM(fee) AS revenue
FROM sessions
GROUP BY therapist_id
"""
revenue_per_therapist_df = run_query(sql_revenue_per_therapist)
print("\nRevenue Per Therapist:")
print(revenue_per_therapist_df)


# therapist bonus if sessions > 10 in a month
sql_monthly_bonus = """
WITH monthly AS (
  SELECT therapist_id,
         strftime('%Y-%m', session_date) AS month,
         COUNT(*) AS session_count,
         SUM(fee) AS monthly_revenue
  FROM sessions
  GROUP BY therapist_id, month
)
SELECT therapist_id, month, session_count, monthly_revenue,
       ROUND(0.20 * monthly_revenue, 2) AS bonus
FROM monthly
WHERE session_count > 10
"""
monthly_bonus_df = run_query(sql_monthly_bonus)
print("\nMonthly Bonus for Therapists with >10 sessions:")
print(monthly_bonus_df)

Total Revenue:
   total_revenue
0         803500

Revenue Per User:
     user_id  revenue_per_user
0          1              1000
1          2               700
2          3               500
3          4              2200
4          5              1700
..       ...               ...
495      496               500
496      497               700
497      498               500
498      499               500
499      500               500

[500 rows x 2 columns]

Revenue Per Cohort:
  cohort_month  revenue
0      2025-01    97300
1      2025-02   126000
2      2025-03   117100
3      2025-04   143400
4      2025-05   153600
5      2025-06   135700
6      2025-07    30400

Revenue Per Therapist:
    therapist_id  revenue
0              1    15000
1              2    13000
2              3    18900
3              4    20000
4              5    11800
5              6    20800
6              7    13100
7              8    22300
8              9    20800
9             10    16500
10           

In [ ]:
# checking where users drop off (last session attended)
dropoff_sql = """
WITH last_session AS (
  SELECT user_id, MAX(session_number) AS last_session
  FROM sessions
  GROUP BY user_id
)
SELECT last_session,
       COUNT(*) AS users_dropped,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM last_session
GROUP BY last_session
ORDER BY last_session
"""

dropoff_df = run_query(dropoff_sql)
dropoff_df

,last_session,users_dropped,pct
0,1,219,43.8
1,2,112,22.4
2,3,72,14.4
3,4,49,9.8
4,5,48,9.6


In [ ]:
# saving cleaned tables to Excel so I can upload them later
sessions_df.to_excel("sessions_clean.xlsx", index=False)
users_df.to_excel("users_clean.xlsx", index=False)
feedback_df.to_excel("feedback_clean.xlsx", index=False)

# also exporting important outputs from the SQL analysis
# mainly cohort retention and drop-off distribution
cohorts_df.to_excel("cohort_retention.xlsx", index=False)
dropoff_df.to_excel("dropoff.xlsx", index=False)